In [0]:
from pyspark.sql.functions import col, lit, when, create_map, expr, current_timestamp
from pyspark.sql.types import StringType
from delta.tables import DeltaTable  # <--- NEW IMPORT ADDED HERE
import time

print("🚀 Starting Silver Layer (Map Stage) Processing...")

# ===================================================================================
# 1. Pipeline Configuration & Optimizations
# ===================================================================================

dbutils.widgets.text("customer_id", "")
customer_id = dbutils.widgets.get("customer_id").lower()

bronze_schema = f"bronze.{customer_id}"
silver_catalog = "hgi"
silver_schema = "silver"

bucket_path = "s3://hgi-databricks-data-lakehouse-dev/"
base_checkpoint_dir = f"{bucket_path}layers/silver/checkpoints/"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")

# ===================================================================================
# 2. Metadata Dictionary Setup
# ===================================================================================
spark.sql("""
    CREATE TABLE IF NOT EXISTS ingestion_metadata.standard_schemas (
        source_system STRING, object_name STRING, standard_columns ARRAY<STRING>
    ) USING DELTA
""")

if spark.table("ingestion_metadata.standard_schemas").count() == 0:
    spark.sql("""
        INSERT INTO ingestion_metadata.standard_schemas VALUES
        ('salesforce', 'account', array('Name','AccountNumber','OwnerId','Site','AccountSource','AnnualRevenue','BillingAddress','CleanStatus','CreatedById','DandbCompanyId','DunsNumber','Jigsaw','Description','Tier','NumberOfEmployees','Fax','Industry','LastModifiedById','NaicsCode','NaicsDesc','OperatingHoursId','Ownership','ParentId','Phone','Rating','ShippingAddress','Sic','SicDesc','TickerSymbol','Tradestyle','Type','Website','YearStarted')),
        ('salesforce', 'contact', array('AccountId','AssistantName','AssistantPhone','Birthdate','BuyerAttributes','CleanStatus','OwnerId','ContactSource','Jigsaw','Department','Description','DoNotCall','Email','HasOptedOutOfEmail','Fax','HasOptedOutOfFax','GenderIdentity','HomePhone','IndividualId','LastModifiedById','LastCURequestDate','LastCUUpdateDate','LeadSource','MailingAddress','MobilePhone','Name','OtherAddress','OtherPhone','Phone','Pronouns','ReportsToId','Title')),
        ('salesforce', 'opportunity', array('AccountId','Amount','CloseDate','ContractId','CreatedById','Description','ExpectedRevenue','ForecastCategoryName','LastModifiedById','LeadSource','NextStep','Name','OwnerId','IqScore','Pricebook2Id','CampaignId','IsPrivate','Probability','TotalOpportunityQuantity','StageName','Type')),
        ('salesforce', 'task', array('WhoId','WhatId','Subject','ActivityDate','Status','Priority','IsHighPriority','OwnerId','Description','IsDeleted','AccountId','IsClosed','CallDurationInSeconds','CallType','TaskSubtype')),
        ('salesforce', 'campaign', array('IsActive','Name','Type','Status','StartDate','EndDate','ExpectedRevenue','BudgetedCost','ActualCost','ExpectedResponse','NumberSent')),
        ('salesforce', 'campaignmember', array('CampaignId','LeadId','ContactId','Status','HasResponded')),
        ('bigquery', 'event', array('event','event_text','event_timestamp'))
    """)

def get_standard_columns(source_sys, obj_name):
    try:
        schema_df = spark.sql(f"SELECT standard_columns FROM ingestion_metadata.standard_schemas WHERE source_system = '{source_sys}' AND object_name = '{obj_name}'")
        rows = schema_df.collect()
        if rows: return [c.lower() for c in rows[0]["standard_columns"]]
    except Exception: pass
    return []

# ===================================================================================
# 3. Target Silver Table Initialization 
# ===================================================================================
def initialize_silver_tables():
    base_tracking = "tenant BIGINT, id STRING, source_system STRING, source_system_object STRING, source_key_name STRING, source_key_value STRING, data_timestamp TIMESTAMP, status STRING"
    
    tables = {
        "unified_accounts": f"{base_tracking}, domain STRING, name STRING, custom_metadata MAP<STRING, STRING>",
        "unified_contacts": f"{base_tracking}, email STRING, domain STRING, a_accountid STRING, custom_metadata MAP<STRING, STRING>",
        "unified_opportunities": f"{base_tracking}, a_accountid STRING, a_amount STRING, a_stagename STRING, custom_metadata MAP<STRING, STRING>",
        "unified_tasks": f"{base_tracking}, contact_source_system_object STRING, contact_source_key_value STRING, a_subject STRING, event_timestamp TIMESTAMP, custom_metadata MAP<STRING, STRING>",
        "unified_campaigns": f"{base_tracking}, custom_metadata MAP<STRING, STRING>",
        "unified_campaign_members": f"{base_tracking}, campaign_source_key_value STRING, contact_source_key_value STRING, custom_metadata MAP<STRING, STRING>",
        "unified_events": f"{base_tracking}, event STRING, event_text STRING, event_timestamp TIMESTAMP, domains STRING, custom_metadata MAP<STRING, STRING>"
    }
    
    for table, schema in tables.items():
        cluster_key = "(tenant, domain)" if table == "unified_accounts" else "(tenant, id)"
        
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {silver_catalog}.{silver_schema}.{table} ({schema}) 
            USING DELTA 
            CLUSTER BY {cluster_key}
            TBLPROPERTIES (
                'delta.enableDeletionVectors' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true'
            )
        """)

initialize_silver_tables()

# ===================================================================================
# 4. Core CDC to Silver Merge Logic 
# ===================================================================================
def process_silver_batch(microBatchDF, batchId, source_sys, object_name, target_table):
    if microBatchDF.isEmpty(): return
    df_changes = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))
    if df_changes.isEmpty(): return

    std_cols = get_standard_columns(source_sys, object_name)
    
    # Notice domains added here to support the unified_events physical columns
    core_tracking = ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'data_timestamp', 'status', 'domain', 'domains', 'name', 'email', 'contact_source_system_object', 'contact_source_key_value', 'campaign_source_key_value', 'event_timestamp', 'a_accountid', 'a_subject', 'a_amount', 'a_stagename', 'event', 'event_text']
    
    cols_to_keep_physical = []
    cols_to_pack_into_map = []
    
    for c in df_changes.columns:
        if c in core_tracking or c.startswith("_"): cols_to_keep_physical.append(c)
        elif c.startswith("a_"):
            if c[2:] in std_cols: cols_to_keep_physical.append(c)
            else: cols_to_pack_into_map.append(c)
            
    if cols_to_pack_into_map:
        map_exprs = []
        for c in cols_to_pack_into_map: map_exprs.extend([lit(c), col(c).cast("string")])
        df_packed = df_changes.withColumn("custom_metadata", create_map(*map_exprs))
    else:
        df_packed = df_changes.withColumn("custom_metadata", expr("map()"))
        
    df_final = df_packed.select(*[c for c in cols_to_keep_physical if c in df_packed.columns] + ["custom_metadata"])
    
    target_table_full = f"{silver_catalog}.{silver_schema}.{target_table}"
    
    target_cols_raw = spark.sql(f"SHOW COLUMNS IN {target_table_full}").collect()
    target_cols_lower = [r[0].lower() for r in target_cols_raw]

    new_cols = []
    for f in df_final.schema:
        if f.name.lower() not in target_cols_lower and not f.name.startswith("_"):
            new_cols.append(f"`{f.name}` {f.dataType.simpleString()}")

    if new_cols:
        print(f"✨ Evolving Silver Schema: Adding {len(new_cols)} columns to {target_table}")
        spark.sql(f"ALTER TABLE {target_table_full} ADD COLUMNS ({', '.join(new_cols)})")
    
    # --- FIX APPLIED HERE ---
    # Replaced temporary views with the thread-safe native DeltaTable Python API. 
    # Logic remains exactly identical, output remains exactly identical!
    merge_cols = [c for c in df_final.columns if not c.startswith("_")]
    
    update_dict = {c: f"source.{c}" for c in merge_cols if c not in ['tenant', 'id']}
    insert_dict = {c: f"source.{c}" for c in merge_cols}
    
    delta_table = DeltaTable.forName(spark, target_table_full)
    
    delta_table.alias("target").merge(
        df_final.alias("source"),
        "target.id = source.id AND target.tenant = source.tenant"
    ).whenMatchedUpdate(
        condition="source.data_timestamp >= target.data_timestamp",
        set=update_dict
    ).whenNotMatchedInsert(
        values=insert_dict
    ).execute()

# ===================================================================================
# 5. Start CDF Streams
# ===================================================================================
def start_silver_stream(source_sys, obj_name, target_table):
    target_table_mapping = {
        "account": "crm_accounts", 
        "contact": "crm_contacts",
        "opportunity": "crm_opportunities", 
        "campaign": "crm_campaigns", 
        "user": "crm_users",
        "event": "crm_events", 
        "task": "crm_tasks",
        "campaign_member": "crm_campaign_members"
    }
    
    bronze_physical_name = target_table_mapping.get(obj_name, f"crm_{obj_name}s")
    bronze_table = f"{bronze_schema}.{bronze_physical_name}"
    
    spark.sql(f"ALTER TABLE {bronze_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    
    checkpoint_location = f"{base_checkpoint_dir}{source_sys}/{customer_id}/{obj_name}/"
    
    stream = spark.readStream \
        .format("delta") \
        .option("readChangeFeed", "true") \
        .table(bronze_table)
        
    query = stream.writeStream.foreachBatch(lambda df, epochId: process_silver_batch(df, epochId, source_sys, obj_name, target_table)) \
        .option("checkpointLocation", checkpoint_location) \
        .trigger(availableNow=True).start()
        
    return query

print("Streaming Bronze CDC to Silver Unified Tables...")
queries = [
    start_silver_stream("salesforce", "account", "unified_accounts"),
    start_silver_stream("salesforce", "contact", "unified_contacts"),
    start_silver_stream("salesforce", "opportunity", "unified_opportunities"),
    start_silver_stream("salesforce", "task", "unified_tasks"),
    start_silver_stream("salesforce", "campaign", "unified_campaigns"),
    start_silver_stream("salesforce", "campaign_member", "unified_campaign_members"),
    start_silver_stream("bigquery", "event", "unified_events")
]

for q in queries: q.awaitTermination()
print("✅ Base Silver CDC Merge Complete. Proceeding to Map Derivations...")

# ===================================================================================
# 6. Map Layer Derivations (Compliant with Output Schemas Summary & Flowcharts)
# ===================================================================================
print("🔄 Executing Map Layer Analytical Derivations...")

# 1.2 Refresh Materialized Views
spark.sql(f"""
    CREATE OR REPLACE VIEW {silver_catalog}.{silver_schema}.contacts_all AS
    SELECT id, email, domain, source_system, tenant FROM {silver_catalog}.{silver_schema}.unified_contacts
""")
spark.sql(f"""
    CREATE OR REPLACE VIEW {silver_catalog}.{silver_schema}.accounts_all AS
    SELECT id, domain, source_system, tenant FROM {silver_catalog}.{silver_schema}.unified_accounts
""")

# 1.3 CRM Events Mapping
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.crm_events USING DELTA AS
    SELECT 
        id, tenant, event_timestamp, 
        concat('salesforce_', contact_source_system_object, '_Id_', contact_source_key_value) AS contact_id,
        CASE 
            WHEN a_subject = 'Call' THEN 'call'
            WHEN a_subject LIKE 'Email: Re:%' THEN 'email_reply'
            WHEN a_subject LIKE 'Email:%' THEN 'email_sent'
            ELSE 'other'
        END AS meta_event
    FROM {silver_catalog}.{silver_schema}.unified_tasks
    WHERE source_system = 'salesforce' AND contact_source_key_value IS NOT NULL
""")

# 1.4 Events Mapping (FIXED: Resolves contactid correctly to fix empty downstream tables) 
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.events USING DELTA AS
    -- Native External Events (BigQuery)
    SELECT 
        e.id, e.tenant, e.event_timestamp, 
        COALESCE(e.event, 'other') AS meta_event, 
        COALESCE(c.id, concat(e.source_system, '_Contact_Id_', e.contact_source_key_value)) AS contactid,
        e.domains AS domain
    FROM {silver_catalog}.{silver_schema}.unified_events e
    LEFT JOIN {silver_catalog}.{silver_schema}.contacts_all c 
        ON c.email = e.contact_source_key_value 
        OR c.id = concat(e.source_system, '_Contact_Id_', e.contact_source_key_value)
        
    UNION ALL
    
    -- CRM Mapped Events (From Salesforce Tasks)
    SELECT 
        ce.id, ce.tenant, ce.event_timestamp, 
        ce.meta_event, 
        ce.contact_id AS contactid,
        c.domain AS domain
    FROM {silver_catalog}.{silver_schema}.crm_events ce
    LEFT JOIN {silver_catalog}.{silver_schema}.contacts_all c ON ce.contact_id = c.id
""")

# 1.5 Contacts-to-Accounts (FIXED: 3-Phase Logic creates accounts per flowchart)
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.contacts_to_accounts USING DELTA AS
    -- Phase 1: CRM Defined
    SELECT c.id AS contact_id, concat('salesforce_Account_Id_', c.a_accountid) AS account_id, 'crm_defined' AS match_type
    FROM {silver_catalog}.{silver_schema}.unified_contacts c 
    WHERE c.a_accountid IS NOT NULL
    
    UNION ALL
    
    -- Phase 2 & 3: Domain and Email Matching (Generates synthetic accounts)
    SELECT 
        c.id AS contact_id, 
        CASE 
            WHEN c.domain IN ('gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com') THEN concat('madkudu_map_email_', c.email)
            WHEN a.id IS NOT NULL THEN a.id
            ELSE concat('madkudu_map_domain_', c.domain)
        END AS account_id,
        CASE 
            WHEN c.domain IN ('gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com') THEN 'free_email_match'
            WHEN a.id IS NOT NULL THEN 'domain_match'
            ELSE 'domain_created'
        END AS match_type
    FROM {silver_catalog}.{silver_schema}.unified_contacts c
    LEFT JOIN (
        -- Fetch first account for a domain to avoid duplication
        SELECT domain, FIRST(id) as id FROM {silver_catalog}.{silver_schema}.unified_accounts WHERE domain IS NOT NULL GROUP BY domain
    ) a ON c.domain = a.domain
    WHERE c.a_accountid IS NULL AND c.domain IS NOT NULL
""")

# 1.6 Event Aggregations (Email and Domain Level)
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.email_events_mapped USING DELTA AS
    SELECT c.email, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences
    FROM {silver_catalog}.{silver_schema}.events e
    JOIN {silver_catalog}.{silver_schema}.contacts_all c ON e.contactid = c.id
    WHERE c.email IS NOT NULL
    GROUP BY c.email, e.meta_event, CAST(e.event_timestamp AS DATE)
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.domain_events_mapped USING DELTA AS
    SELECT c.domain, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences
    FROM {silver_catalog}.{silver_schema}.events e
    JOIN {silver_catalog}.{silver_schema}.contacts_all c ON e.contactid = c.id
    WHERE c.domain IS NOT NULL
    GROUP BY c.domain, e.meta_event, CAST(e.event_timestamp AS DATE)
""")

# NEW: 1.6 Account-Level Rollup Table
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.mk_account_events_mapped USING DELTA AS
    -- Identified Events
    SELECT 
        c2a.account_id AS mk_accountid_domain, 
        e.meta_event, 
        CAST(e.event_timestamp AS DATE) AS event_day, 
        COUNT(*) AS occurrences,
        false AS anonymous_activity
    FROM {silver_catalog}.{silver_schema}.events e
    JOIN {silver_catalog}.{silver_schema}.contacts_to_accounts c2a ON e.contactid = c2a.contact_id
    WHERE e.contactid IS NOT NULL
    GROUP BY 1, 2, 3, 5
    
    UNION ALL
    
    -- Anonymous Events (domain matched, no contact attached)
    SELECT 
        a.id AS mk_accountid_domain, 
        e.meta_event, 
        CAST(e.event_timestamp AS DATE) AS event_day, 
        COUNT(*) AS occurrences,
        true AS anonymous_activity
    FROM {silver_catalog}.{silver_schema}.events e
    JOIN {silver_catalog}.{silver_schema}.accounts_all a ON e.domain = a.domain
    WHERE e.contactid IS NULL AND e.domain IS NOT NULL
    GROUP BY 1, 2, 3, 5
""")

# NEW: Attributes Tables
spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.accounts_attributes USING DELTA AS
    SELECT id AS account_id, false AS is_paying, false AS is_excluded, 0.0 AS mrr 
    FROM {silver_catalog}.{silver_schema}.accounts_all
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {silver_catalog}.{silver_schema}.contacts_attributes USING DELTA AS
    SELECT id AS contact_id, false AS is_paying, false AS is_excluded, 0.0 AS mrr 
    FROM {silver_catalog}.{silver_schema}.contacts_all
""")

# 1.7 Audiences & Conversion (Optional for POC, initializing safely)
spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_catalog}.{silver_schema}.email_audience (email STRING, audience_name STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_catalog}.{silver_schema}.email_model_conversion (email STRING, conversion_model STRING, conversion_date DATE, amount DOUBLE) USING DELTA")

print("✅ Map Layer Analytical Derivations Complete!")